In [ ]:
#

In [ ]:
'''import shutil

# Apaga a pasta inteira (mesmo se tiver arquivos dentro)
shutil.rmtree('/content/Classification')
'''

"import shutil\n\n# Apaga a pasta inteira (mesmo se tiver arquivos dentro)\nshutil.rmtree('/content/Classification')\n"

In [ ]:
'''import shutil

# Compacta a pasta em um arquivo zip
shutil.make_archive('/content/Classification_zip', 'zip', '/content/Classification')

# Baixa o arquivo zip
from google.colab import files
files.download('/content/Classification_zip.zip')'''


"import shutil\n\n# Compacta a pasta em um arquivo zip\nshutil.make_archive('/content/Classification_zip', 'zip', '/content/Classification')\n\n# Baixa o arquivo zip\nfrom google.colab import files\nfiles.download('/content/Classification_zip.zip')"

In [ ]:
import random

import numpy as np
import json
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import os
import time
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, cohen_kappa_score
from sklearn.metrics import ConfusionMatrixDisplay

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

class Results:
    accuracy = pd.DataFrame()
    precision = pd.DataFrame()
    recall = pd.DataFrame()
    f1 = pd.DataFrame()
    kappa = pd.DataFrame()
    train_accuracy = pd.DataFrame()
    train_precision = pd.DataFrame()
    train_recall = pd.DataFrame()
    train_f1 = pd.DataFrame()
    train_kappa = pd.DataFrame()
    cv_mean = pd.DataFrame()
    cv_stddev = pd.DataFrame()
    train_error_rate = pd.DataFrame()
    gap = pd.DataFrame()
    error_rate = pd.DataFrame()
    train_time = pd.DataFrame()
    infer_time = pd.DataFrame()

    accuracy["Setup"] = ""
    precision["Setup"] = ""
    recall["Setup"] = ""
    f1["Setup"] = ""
    cv_mean["Setup"] = ""
    cv_stddev["Setup"] = ""
    kappa["Setup"] = ""
    error_rate["Setup"] = ""
    train_time["Setup"] = ""
    infer_time["Setup"] = ""
    train_accuracy["Setup"] = ""
    train_precision["Setup"] = ""
    train_recall["Setup"] = ""
    train_f1["Setup"] = ""
    train_kappa["Setup"] = ""
    train_error_rate["Setup"] = ""
    gap["Setup"] = ""

    accuracy.set_index('Setup')
    precision.set_index('Setup')
    recall.set_index('Setup')
    f1.set_index('Setup')
    cv_mean.set_index('Setup')
    cv_stddev.set_index('Setup')
    kappa.set_index('Setup')
    error_rate.set_index('Setup')
    train_time.set_index('Setup')
    infer_time.set_index('Setup')
    train_accuracy.set_index('Setup')
    train_precision.set_index('Setup')
    train_recall.set_index('Setup')
    train_f1.set_index('Setup')
    train_kappa.set_index('Setup')
    train_error_rate.set_index('Setup')
    gap.set_index('Setup')

    def addModel(self, name):
        self.accuracy[name] = ""
        self.precision[name] = ""
        self.recall[name] = ""
        self.f1[name] = ""
        self.cv_mean[name] = ""
        self.cv_stddev[name] = ""
        self.kappa[name] = ""
        self.error_rate[name] = ""
        self.train_time[name] = ""
        self.infer_time[name] = ""
        self.train_accuracy[name] = ""
        self.train_precision[name] = ""
        self.train_recall[name] = ""
        self.train_f1[name] = ""
        self.train_kappa[name] = ""
        self.train_error_rate[name] = ""
        self.gap[name] = ""

    def addResults(self, _acc, _precision, _recall, _f1, _cv_mean, _cv_stddev, _kappa, _error_rate, _train_time, _infer_time, _train_acc, _train_precision, _train_recall, _train_f1, _train_kappa, _train_error_rate, _gap):
        self.accuracy.loc[len(self.accuracy)] = _acc
        self.precision.loc[len(self.precision)] = _precision
        self.recall.loc[len(self.recall)] = _recall
        self.f1.loc[len(self.f1)] = _f1
        self.cv_mean.loc[len(self.cv_mean)] = _cv_mean
        self.cv_stddev.loc[len(self.cv_stddev)] = _cv_stddev
        self.kappa.loc[len(self.kappa)] = _kappa
        self.error_rate.loc[len(self.error_rate)] = _error_rate
        self.train_time.loc[len(self.train_time)] = _train_time
        self.infer_time.loc[len(self.infer_time)] = _infer_time
        self.train_accuracy.loc[len(self.accuracy)] = _train_acc
        self.train_precision.loc[len(self.precision)] = _train_precision
        self.train_recall.loc[len(self.recall)] = _train_recall
        self.train_f1.loc[len(self.f1)] = _train_f1
        self.train_kappa.loc[len(self.kappa)] = _train_kappa
        self.train_error_rate.loc[len(self.error_rate)] = _train_error_rate
        self.gap.loc[len(self.gap)] = _gap

    def printCSV(self, output_folder="Classification"):
      # Cria a pasta com o subdiretório
        full_path = os.path.join("Classification", output_folder)
        os.makedirs(full_path, exist_ok=True)
        final_results = pd.DataFrame()

        final_results["Accuracy"] = self.accuracy.mean(numeric_only=True)
        final_results["Precision"] = self.precision.mean(numeric_only=True)
        final_results["Recall"] = self.recall.mean(numeric_only=True)
        final_results["F1"] = self.f1.mean(numeric_only=True)
        final_results["CV_Mean"] = self.cv_mean.mean(numeric_only=True)
        final_results["CV_Std"] = self.cv_stddev.mean(numeric_only=True)
        final_results["Kappa"] = self.kappa.mean(numeric_only=True)
        final_results["Error_Rate"] = self.error_rate.mean(numeric_only=True)
        final_results["Train_Time"] = self.train_time.mean(numeric_only=True)
        final_results["Infer_Time"] = self.infer_time.mean(numeric_only=True)

        # treino
        final_results["Train_Accuracy"] = self.train_accuracy.mean(numeric_only=True)
        final_results["Train_Precision"] = self.train_precision.mean(numeric_only=True)
        final_results["Train_Recall"] = self.train_recall.mean(numeric_only=True)
        final_results["Train_F1"] = self.train_f1.mean(numeric_only=True)
        final_results["Train_Kappa"] = self.train_kappa.mean(numeric_only=True)
        final_results["Train_Error_Rate"] = self.train_error_rate.mean(numeric_only=True)

        # gap
        final_results["Gap"] = self.gap.mean(numeric_only=True)

        final_results.reset_index(inplace=True)
        final_results.rename(columns={"index": "Modelo"}, inplace=True)

        final_results.to_csv(
            f"{full_path}/classification_all_results.csv",
            index=False,
            float_format="%.4f"
        )

def initialize_data_frame():
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    sns.set(rc={'figure.figsize':(19.20, 10.80)})


def get_dungeon_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['dungeons']).T


def get_user_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['users']).T


def drop_invalid_rows(_data_frame):
    _data_frame.drop(_data_frame[_data_frame['UniqueRoomsEntered'] == 0].index, inplace=True)
    return _data_frame


def drop_unused_columns(_data_frame):
    _data_frame.drop('__collections__', inplace=True, axis=1)
    _data_frame = _data_frame.drop(list(_data_frame.filter(regex='^Total.*Quests')), axis=1)
    _data_frame.drop('CompletedQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedKillQuests', inplace=True, axis=1)
    _data_frame.rename(columns={'CompletedDamageQuests': 'CompletedKillQuests'}, inplace=True)
    _data_frame.drop('TreasuresCollected', inplace=True, axis=1)
    _data_frame.drop('TotalNpcs', inplace=True, axis=1)
    _data_frame.drop('WeaponName', inplace=True, axis=1)
    _data_frame.drop('HeatMap', inplace=True, axis=1)
    return _data_frame


def get_percentages(_data_frame):
    _data_frame['LockUsage'] = _data_frame['LocksOpened'].div(_data_frame['TotalLocks'].values)*100
    _data_frame['KeyCollection'] = _data_frame['KeysCollected'].div(_data_frame['TotalKeys'].values)*100
    _data_frame['MapCompletion'] = _data_frame['UniqueRoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['RoomRevisitRate'] = _data_frame['RoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['EnemyKillRate'] = _data_frame['EnemiesKilled'].div(_data_frame['TotalEnemies'].values)*100
    _data_frame['TotalAttempts'] = _data_frame['TotalWins'] + _data_frame['TotalDeaths'].values
    _data_frame['TotalAttempts'] = _data_frame['TotalAttempts'].astype(float)
    _data_frame['TotalWins'] = _data_frame['TotalWins'].astype(float)
    _data_frame['Success Rate'] = _data_frame['TotalAttempts'].div(_data_frame['TotalWins'].values)*100
    _data_frame.replace([np.inf, -np.inf], 0, inplace=True)
    return _data_frame


def drop_processed_columns(_data_frame):
    _data_frame.drop('LocksOpened', inplace=True, axis=1)
    _data_frame.drop('TotalLocks', inplace=True, axis=1)
    _data_frame.drop('KeysCollected', inplace=True, axis=1)
    _data_frame.drop('TotalKeys', inplace=True, axis=1)
    _data_frame.drop('EnemiesKilled', inplace=True, axis=1)
    _data_frame.drop('TotalEnemies', inplace=True, axis=1)
    _data_frame.drop('TotalAttempts', inplace=True, axis=1)
    _data_frame.drop('TotalWins', inplace=True, axis=1)
    _data_frame.drop('TotalDeaths', inplace=True, axis=1)
    _data_frame.drop('NpcsInteracted', inplace=True, axis=1)
    _data_frame.drop('RoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalRooms', inplace=True, axis=1)
    _data_frame.drop('UniqueRoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalTreasure', inplace=True, axis=1)
    return _data_frame


def split_preform_answers(_data_frame):
    _data_frame[['A', 'C', 'I', 'M']] = _data_frame['PreFormAnswers'].str.split(',', expand=True)
    _data_frame['A'] = _data_frame['A'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['C'] = _data_frame['C'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['I'] = _data_frame['I'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['M'] = _data_frame['M'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    return _data_frame


def drop_quest_data(_data_frame):
    _data_frame = _data_frame.drop(list(_data_frame.filter(regex='Quest')), axis=1)
    return _data_frame


def drop_player_data(_data_frame):
    _data_frame.drop('A', inplace=True, axis=1)
    _data_frame.drop('C', inplace=True, axis=1)
    _data_frame.drop('I', inplace=True, axis=1)
    _data_frame.drop('M', inplace=True, axis=1)
    _data_frame.drop('PostFormAnswers', inplace=True, axis=1)
    _data_frame.drop('PlayerId', inplace=True, axis=1)
    return _data_frame


def drop_gameplay_data(_data_frame):
    _data_frame.drop('TimeToFinish', inplace=True, axis=1)
    _data_frame.drop('LockUsage', inplace=True, axis=1)
    _data_frame.drop('KeyCollection', inplace=True, axis=1)
    _data_frame.drop('MapCompletion', inplace=True, axis=1)
    _data_frame.drop('EnemyKillRate', inplace=True, axis=1)
    return _data_frame


def drop_terminal_quest_data(_data_frame):
    _data_frame.drop('CompletedReportQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedListenQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedExploreQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedExchangeQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedKillQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedGiveQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedGoToQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedGatherQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedReadQuests', inplace=True, axis=1)
    return _data_frame


def augment_dataframe_with_scaling(df, y, n, scale_factor):
    """
    Augments the DataFrame by adding `n` new rows, scaling numeric columns by a given factor.
    The new rows inherit the same 'y' values as their corresponding sampled rows.

    Parameters:
        df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        n (int): Number of new rows to generate.
        scale_factor (float): The factor by which numeric columns will be scaled.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify numeric columns, excluding 'y'
    numeric_columns = df.select_dtypes(include=['number']).columns.difference(y)
    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = df.sample(n=n, replace=True).reset_index(drop=True)

    # Scale numeric columns
    for col in numeric_columns:
        sampled_rows[col] *= scale_factor

    # Append the scaled rows to the original DataFrame
    augmented_df = pd.concat([df, sampled_rows], ignore_index=True)

    return augmented_df


def augment_data_random(_df, y, n):
    """
    Augments the DataFrame by adding `n` new rows.
    - Adds noise to numerical columns using a normal distribution.
    - Retains the same 'y' values for the newly added rows as their corresponding sampled rows.

    Parameters:
        _df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        y (list): the column(s) that will have their value retained
        n (int): Number of new rows to generate.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify float and integer columns, excluding 'y'
    float_columns = _df.select_dtypes(include=['float']).columns.difference(y)
    int_columns = _df.select_dtypes(include=['integer']).columns.difference(y)

    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = _df.sample(n=n, replace=True).reset_index(drop=True)

    # Add noise to float columns
    for col in float_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    for col in int_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    # Ensure integer columns remain integers
    sampled_rows[int_columns] = sampled_rows[int_columns].round().astype(int)

    # Append the sampled (augmented) rows to the original DataFrame
    augmented_df = pd.concat([_df, sampled_rows], ignore_index=True)
    return augmented_df

def recommend_from_other_user(target_user, other_user, user_item_matrix):

    print(f"\nRecomendando para usuário {target_user} com base no usuário {other_user}")

    # Itens que target_user já jogou
    target_items = user_item_matrix.loc[target_user]

    # Itens que other_user já jogou
    other_items = user_item_matrix.loc[other_user]

    # Pegar apenas itens que:
    # - other_user jogou (rating > 0)
    # - target_user NÃO jogou (rating == 0)
    candidate_items = other_items[
        (other_items > 0) & (target_items == 0)
    ]

    if len(candidate_items) == 0:
        print("Nenhum item novo para recomendar.")
        return

    # Ordenar pelos melhores ratings do other_user
    recommendations = candidate_items.sort_values(ascending=False)

    print("\nItens recomendados:")
    print(recommendations.head(5))

# ---------------- RECOMENDAÇÃO USER-BASED COMPLETA ---------------- #

def recommend_user_based(target_user, user_item_matrix, user_similarity_df, top_n=5):

    print(f"\nRecomendações para usuário {target_user}")

    # Similaridade do usuário alvo com todos os outros
    similarities = user_similarity_df.loc[target_user]

    # Remover ele mesmo
    similarities = similarities.drop(target_user)

    # Pegar itens que o usuário ainda NÃO jogou
    user_ratings = user_item_matrix.loc[target_user]
    items_not_played = user_ratings[user_ratings == 0].index

    predictions = {}

    for item in items_not_played:

        # Usuários que avaliaram esse item
        users_who_rated = user_item_matrix[user_item_matrix[item] > 0].index

        numerator = 0
        denominator = 0

        for other_user in users_who_rated:

            sim = similarities.get(other_user, 0)

            if sim > 0:  # só similaridade positiva
                rating = user_item_matrix.loc[other_user, item]
                numerator += sim * rating
                denominator += abs(sim)

        if denominator > 0:
            predicted_rating = numerator / denominator
            predictions[item] = predicted_rating

    if len(predictions) == 0:
        print("Nenhum item possível para recomendar.")
        return

    recommendations = (
        pd.Series(predictions)
        .sort_values(ascending=False)
        .head(top_n)
    )

    print("\nItens recomendados:")
    print(recommendations)

# ---------------- RECOMENDAÇÃO USER-BASED EXPLICÁVEL ---------------- #

def recommend_user_based_explain(target_user, user_item_matrix, user_similarity_df, top_n=5):

    print(f"\nRecomendações para usuário {target_user}")

    similarities = user_similarity_df.loc[target_user].drop(target_user)
    user_ratings = user_item_matrix.loc[target_user]
    items_not_played = user_ratings[user_ratings == 0].index

    predictions = {}
    explanations = {}

    for item in items_not_played:

        users_who_rated = user_item_matrix[user_item_matrix[item] > 0].index

        numerator = 0
        denominator = 0
        contributors = []

        for other_user in users_who_rated:

            sim = similarities.get(other_user, 0)

            if sim > 0:
                rating = user_item_matrix.loc[other_user, item]
                contribution = sim * rating

                numerator += contribution
                denominator += abs(sim)

                contributors.append({
                    "user": other_user,
                    "similarity": sim,
                    "rating": rating,
                    "contribution": contribution
                })

        if denominator > 0:
            predicted_rating = numerator / denominator
            predictions[item] = predicted_rating
            explanations[item] = contributors

    if len(predictions) == 0:
        print("Nenhum item possível para recomendar.")
        return

    recommendations = (
        pd.Series(predictions)
        .sort_values(ascending=False)
        .head(top_n)
    )

    print("\nItens recomendados:")
    print(recommendations)

    # Mostrar explicação detalhada
    for item in recommendations.index:
        print("\n---------------------------------------")
        print(f"Explicação para {item}")
        print("---------------------------------------")

        contribs = sorted(
            explanations[item],
            key=lambda x: x["contribution"],
            reverse=True
        )

        for c in contribs[:5]:  # top 5 contribuidores
            print(
                f"Usuário {c['user']} | "
                f"similaridade={round(c['similarity'],3)} | "
                f"nota={round(c['rating'],2)} | "
                f"contribuição={round(c['contribution'],3)}"
            )
# Recommend items similar to 'Item1' #TODO VERIFICAR FUNÇÃO
def recommend_similar_items(item, similarity_df, top_n=3):
  return similarity_df[item].sort_values(ascending=False)[1:top_n+1]

def print_user_ratings(user_item_matrix, user_similarity_df, user1_idx=0, user2_idx=1):
    # seleciona usuários
    u1 = user_similarity_df.index[user1_idx]
    u2 = user_similarity_df.index[user2_idx]

    print("Notas do usuário 1:")
    print(user_item_matrix.loc[u1].dropna())

    print("\nNotas do usuário 2:")
    print(user_item_matrix.loc[u2].dropna())

def analyze_user_activity(user_item_matrix):
    # Contar quantos níveis cada usuário jogou
    levels_per_user = user_item_matrix.notna().sum(axis=1)

    # Filtrar usuários que jogaram mais de 1 nível
    users_more_than_one = levels_per_user[levels_per_user > 1]

    print("Usuários que jogaram mais de um nível:")
    print(users_more_than_one.head())

    # Pegar o primeiro deles
    if len(users_more_than_one) > 0:
        u = users_more_than_one.index[0]

        print("\nUsuário selecionado:", u)
        print("\nQuantidade de níveis jogados:", levels_per_user.loc[u])

        print("\nNotas desse usuário:")
        print(user_item_matrix.loc[u].dropna())
    else:
        print("Nenhum usuário jogou mais de um nível.")

    # Distribuição geral
    print("\nDistribuição:")
    print(levels_per_user.value_counts().sort_index())

def build_quest_df(df):

    quest_cols = [col for col in df.columns if "Completed" in col and "Quests" in col]

    quest_df = df[['PlayerId'] + quest_cols].copy()

    # transformar em formato longo
    quest_df = quest_df.melt(
        id_vars='PlayerId',
        var_name='Item',
        value_name='Rating'
    )

    # remover zeros
    quest_df = quest_df[quest_df['Rating'] > 0]

    quest_df.columns = ['User', 'Item', 'Rating']

    return quest_df

def build_enemy_df(df):

    df = df.copy()

    # dificuldade simples (você pode melhorar depois)
    df['EnemyDifficulty'] = pd.cut(
        df['TotalEnemies'],
        bins=3,
        labels=['Easy', 'Medium', 'Hard']
    )

    enemy_df = df[['PlayerId', 'EnemyDifficulty', 'EnemiesKilled']].copy()

    enemy_df.columns = ['User', 'Item', 'Rating']

    return enemy_df

def build_struct_df(df):

    df = df.copy()

    df['Item'] = (
        "R" + df['TotalRooms'].astype(str) +
        "_K" + df['TotalKeys'].astype(str) +
        "_RR" + (df['RoomsEntered'] - df['UniqueRoomsEntered']).astype(str)
    )

    df['Rating'] = df['PostFormAnswers'].apply(
        lambda x: sum(x)/len(x) if isinstance(x, list) else x
    )

    struct_df = df[['PlayerId', 'Item', 'Rating']]
    struct_df.columns = ['User', 'Item', 'Rating']

    return struct_df



def collaborative_filtering(_df, method):
    # ------------------------------------------------- #
    # 1- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
    # *[OK]
    # 2- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
    # *[OK]
    # 3- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
    # *[OK]
    # 4- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
    # *[OK]
    # ------------------------------------------------- #

    # 5- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
    # *[OK]
    # 6- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
    # *[OK]
    # 7- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
    # *[OK]
    # 8- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
    # *[OK]
    # ------------------------------------------------- #

    # 9- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
    # *
    # 10- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
    # *
    # 11 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
    # *
    # 12 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
    # ------------------------------------------------- #
    # TODO: FAZER O GRANULAR, MAS DADOS SEPARADOS SEM SER UMA TABELA // PRA COMPARAR SEPARADMENTES (VE SE FAZ SENTIDO) - SENAO EXPANDIR O 9 AO 12
    # ANTES DE AVANÇAR ORGANIZAR AQUI
    # ------------- VISUALIZAÇÃO DOS DADOS ------------#
    print(_df.shape)
    print(_df.head())
    print("-"*80)

    print("\nFrequência no dungeon_frame final:")
    print(_df['PreFormAnswers'].value_counts())

    # Número de usuários únicos pelo PlayerId
    num_unicos = _df['PlayerId'].nunique()
    print(f"\nNúmero de usuários únicos: {num_unicos}")

    # ------------- NORMALIZAÇÃO DOS DADOS ------------#

    if(method == 1 or method == 2 or method == 5 or method == 6): # dungeons com perfis; Filtragem baseada em itens
      if(method == 1 or method == 2):
        cols = ['PlayerId', 'PreFormAnswers', 'PostFormAnswers']
      else:  # method == 5 or 6
          cols = ['PlayerId', 'LevelName', 'PostFormAnswers']

      collab_df = _df[cols].copy() #Usuário, config. dungeon, avaliação
      collab_df.columns = ['User', 'Item', 'Rating']

      # cálculo da média do PostFormAnswers
      collab_df['Rating'] = collab_df['Rating'].apply(lambda x: sum(x)/len(x) if isinstance(x, list) else x)

    elif(method == 3 or method == 4 or method == 7 or method == 8): # dungeons com perfis; Filtragem baseada em usuario
      if(method == 3 or method == 4):
        cols = ['PlayerId', 'PreFormAnswers', 'Success Rate',
                  'TotalLostHealth',
                  'EnemyKillRate',
                  'MapCompletion']
      else:  # method == 7 or 8
          cols = ['PlayerId', 'LevelName', 'Success Rate',
                  'TotalLostHealth',
                  'EnemyKillRate',
                  'MapCompletion']

      # Filtragem baseada em itens - Nota
      collab_df = _df[cols].copy() #Usuário, config. dungeon, avaliação
      collab_df.columns = ['User', 'Item', 'SuccessRate',
                      'TotalLostHealth',
                      'EnemyKillRate',
                      'MapCompletion']

      # -------------------------------
      # NORMALIZAÇÃO DO TotalLostHealth
      # (min-max scaling)
      # -------------------------------

      min_health = collab_df['TotalLostHealth'].min()
      max_health = collab_df['TotalLostHealth'].max()

      collab_df['TotalLostHealth_norm'] = (
          (collab_df['TotalLostHealth'] - min_health) /
          (max_health - min_health + 1e-9)
      )

    # TODO VER SE O LEO FAZ ESSA conNTA EM ALGUM LUGAR

      collab_df['Rating'] = (
          collab_df['SuccessRate']
          - collab_df['TotalLostHealth_norm']
          + collab_df['EnemyKillRate']
          + collab_df['MapCompletion']
      )

      # -------------------------------
      # Manter apenas colunas finais
      # -------------------------------

      collab_df = collab_df[['User', 'Item', 'Rating']]
      ##TODO FUTURO PODERIA USAR OPERFIL PRA DEIXA RMLEHOIR
    else:
      if(method == 9 or method == 10):
        print("entrou auqi------------")
        #cols = ['PlayerId', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'PostFormAnswers']
        cols = ['PlayerId', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'PostFormAnswers']
      else:  # method == 5 or 6
          cols = ['PlayerId', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'Success Rate',
                  'TotalLostHealth',
                  'EnemyKillRate',
                  'MapCompletion']

      collab_df = _df[cols].copy() #Usuário, config. dungeon, avaliação
      if(method == 9 or method == 10):
        collab_df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'Rating']
        collab_df['TotalRooms'] = collab_df['TotalRooms'].replace(0, 1)

        # Calcular Item
        collab_df['Item'] = (
            "R" + collab_df['TotalRooms'].astype(str) +
            "_K" + collab_df['TotalKeys'].astype(str) +
            "_RR" + (collab_df['RoomsEntered'] - collab_df['UniqueRoomsEntered']).astype(str)
        )

        collab_df['Rating'] = collab_df['Rating'].apply(
            lambda x: sum(x)/len(x) if isinstance(x, list) else x
        )

      else:
          collab_df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'SuccessRate',
                      'TotalLostHealth',
                      'EnemyKillRate',
                      'MapCompletion']
          # Calcular Item
          collab_df['Item'] = (
              "R" + collab_df['TotalRooms'].astype(str) +
              "_K" + collab_df['TotalKeys'].astype(str) +
              "_RR" + (collab_df['RoomsEntered'] - collab_df['UniqueRoomsEntered']).astype(str)
          )

          # -------------------------------
          # NORMALIZAÇÃO DO TotalLostHealth
          # (min-max scaling)
          # -------------------------------

          min_health = collab_df['TotalLostHealth'].min()
          max_health = collab_df['TotalLostHealth'].max()

          collab_df['TotalLostHealth_norm'] = (
              (collab_df['TotalLostHealth'] - min_health) /
              (max_health - min_health + 1e-9)
          )

        # TODO VER SE O LEO FAZ ESSA conNTA EM ALGUM LUGAR

          collab_df['Rating'] = (
              collab_df['SuccessRate']
              - collab_df['TotalLostHealth_norm']
              + collab_df['EnemyKillRate']
              + collab_df['MapCompletion']
          )

          collab_df = collab_df[['User', 'Item', 'Rating']]

          print("no, non,  nein")

      collab_df = collab_df[['User', 'Item', 'Rating']]




    print("\nMatriz para filtragem 1 :")
    print(collab_df.head())

    # ------------- CÁLCULOS ------------#

    # ?
    collab_f_df = pd.DataFrame(collab_df)

    # Create user-item matrix
    user_item_matrix = collab_f_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)

    # ------------- VERIFICAÇÃO DOS DADOS ------------#
    print("Número de usuários:", user_item_matrix.shape[0])
    print("Número de itens:", user_item_matrix.shape[1])

    print("\nQuantidade média de avaliações por usuário:")
    print(user_item_matrix.count(axis=1).mean())

    print("\nQuantidade média de usuários por item:")
    print(user_item_matrix.count(axis=0).mean())

    # ------------- VARIÂNCIA ITEM POR ITEM ------------#
    print("variancia de cada item")
    print(user_item_matrix.var())

    # ---------------- SIMILIARIDADE
    if(method % 2 == 0):
      # Pearson item-item
      item_similarity_pearson = user_item_matrix.corr(method='pearson')

      print("\nITEM POR ITEM (PEARSON):\n")
      print(item_similarity_pearson)
      item_similarity_df = item_similarity_pearson

      user_similarity_pearson = user_item_matrix.T.corr(method='pearson')
      user_similarity_pearson = user_similarity_pearson.fillna(0)

      print("\nUSUARIO POR USUARIO (PEARSON):\n")
      print(user_similarity_pearson)

      user_similarity_df  = user_similarity_pearson
    else:
      # Compute item similarity using cosine similarity
      item_similarity = cosine_similarity(user_item_matrix.T)
      item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

      #centralizando
      user_item_centered = user_item_matrix.sub(user_item_matrix.mean(axis=1), axis=0)

      # Similaridade entre itens
      # item por item
      item_similarity_centered = cosine_similarity(user_item_centered.T)
      item_similarity_centered_df = pd.DataFrame(item_similarity_centered, index=user_item_centered.columns, columns=user_item_centered.columns)

      print("\nconsseno cetnralizado:\n")
      print(item_similarity_df)

      # usuario por usuário
      user_similarity = cosine_similarity(user_item_centered)
      user_similarity_df = pd.DataFrame( user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index )
      print("\nUSUARIO POR USUARIO:\n")
      print(user_similarity_df)

    # ------------- RECOMENDAR ITEM POR ITEM ------------#

    if(method >=1 and method <=4):
      # Example recommendation
      similar_items = recommend_similar_items('A=4, C=3, I=2, M=1', item_similarity_df)
      print("\nItems similar to A=4, C=3, I=2, M=1:", similar_items)

    # Recomendar todos os itens por similarida itens
    all_recommendations = {}

    for item in user_item_matrix.columns:
        all_recommendations[item] = recommend_similar_items(item, item_similarity_df)

    print(all_recommendations)

    # Recomendar todos os usuário por similarida usuário - TODO ADAPTAR AQUI
    '''all_recommendations_user = {}

    for item in user_item_matrix.columns:
        all_recommendations_user[item] = recommend_similar_items(item, user_similarity_df)

    print(all_recommendations_user)'''

    # ------------- SIMILIARIDA USUARIO POR USUARIO ------------#

    # ENTENDNEO O RESULTADO ACIMA
    print_user_ratings(user_item_matrix, user_similarity_df)

    #TODO: Conferri com o léo é levelname? pq usando o a=2c=4..., nao ta legal, e pelo o que o usaurio joga no msmo nao mais em um consdieradno isso



    # ------------- testando um usuario com mais de um nivel ------------#
    analyze_user_activity(user_item_matrix)

    #todo aima fiz por nota, fazer por comportametno abaixo

    # melhorar codigo depois, para mudar mdos, de entrada e como vai ser feito

    #todo depois fazer considernado os dois

    # todo depois fazr com a entrada granular

    # faze tablea para cada atribulo - inmigos, quest, difuclda, prefenais (maar, explora...)




    ##TOODO FUTURO PODERIA USAR OPERFIL PRA DEIXA RMLEHOIR

    #cheguei ate aqui na organziacao - fazer o abaixo pra todos os modelos,a daptar se necesario

    # Pegando os dois primeiros usuários da matriz
    u1 = user_item_matrix.index[0]
    u2 = user_item_matrix.index[1]

    # Recomendar para u1 baseado em u2
    recommend_from_other_user(u1, u2, user_item_matrix)

    # Recomendar para u2 baseado em u1
    recommend_from_other_user(u2, u1, user_item_matrix)

    # Escolher um usuário qualquer
    target_user = user_item_matrix.index[0]

    recommend_user_based(target_user, user_item_matrix, user_similarity_df, top_n=5)
    #todo recomendaco a aprtir de todos usuarios? ex.: um grupo aprova isso e vc ta no grupo, entao vc vai gostar

    # Escolher usuário
    target_user = user_item_matrix.index[0]

    recommend_user_based_explain(target_user, user_item_matrix, user_similarity_df, top_n=5)
    #todo ordenar recomaencdaos

    #todo organizar codigo tem muita coisa junto ao mesmo tempo

def classify(_df, method):
    #_df.drop('PostFormAnswers', axis=1, inplace=True)
    #_df.drop('PlayerId', axis=1, inplace=True)
    #_df.drop('InputProfile', axis=1, inplace=True)
    #_df.drop('LevelName', axis=1, inplace=True)
    _df['PreFormAnswers'] = _df['PreFormAnswers'].apply(lambda x: 'A=' + str(x[0]) + ", C=" + str(x[1]) + ", I=" + str(x[2]) +
                                                                  ", M=" + str(x[3]))
    _df = split_preform_answers(_df)
    n = _df.shape[0]

    #_df.describe().to_csv('descriptive_statistics.csv')


    _dfnoise = augment_data_random(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfscale = augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfnoise2 = augment_data_random(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 2*n)
    _dfscale2 = augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfscale2 = augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)
    _dfmixed = augment_data_random(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfmixed = pd.concat([_dfmixed, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = augment_data_random(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 6*n)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.10)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.9)], ignore_index=True)

    dataframes = [#_df, _dfnoise, _dfscale, _dfnoise2, _dfscale2, _dfmixed,
                  _dfmixed10]
    experiment_names = [#'Baseline', '1xN', '1xS-1_05', '2xN', '2xS-1_05+0_95', '1xS-1_05+1xN',
                        '6xN+4xS-1_05+0_95+1_1+0_9']

    strategies = [#'Original', '1xNoise', '1xScale', '2xN', '2xScale', '1xN+1xS',
                  '6xN+4xS']

    # Quantos níveis diferentes cada usuário jogou
    levels_per_user = _df.groupby("PlayerId")["LevelName"].nunique()

    # Filtrar apenas quem jogou mais de 1 nível
    multi_level_users = levels_per_user[levels_per_user > 1]

    print("Usuários que jogaram mais de um nível:")
    print(multi_level_users)

    print("\nDetalhamento:\n")

    for user in multi_level_users.index:
        levels = _df[_df["PlayerId"] == user]["LevelName"].unique()
        print(f"Usuário {user} jogou:")
        print(list(levels))
        print("-" * 50)

    collaborative_filtering(_dfmixed10, method)


def reset_results():
    Results.accuracy = pd.DataFrame()
    Results.precision = pd.DataFrame()
    Results.recall = pd.DataFrame()
    Results.f1 = pd.DataFrame()
    Results.kappa = pd.DataFrame()
    Results.train_accuracy = pd.DataFrame()
    Results.train_precision = pd.DataFrame()
    Results.train_recall = pd.DataFrame()
    Results.train_f1 = pd.DataFrame()
    Results.train_kappa = pd.DataFrame()
    Results.cv_mean = pd.DataFrame()
    Results.cv_stddev = pd.DataFrame()
    Results.train_error_rate = pd.DataFrame()
    Results.gap = pd.DataFrame()
    Results.error_rate = pd.DataFrame()
    Results.train_time = pd.DataFrame()
    Results.infer_time = pd.DataFrame()

def main(method):
    initialize_data_frame()
    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()
    dungeon_frame = pd.merge(dungeon_frame, user_frame[['PlayerId', 'PreFormAnswers']], on=['PlayerId'], how='left')
    dungeon_frame = drop_invalid_rows(dungeon_frame)
    dungeon_frame = drop_unused_columns(dungeon_frame)
    dungeon_frame = get_percentages(dungeon_frame)
    #dungeon_frame = drop_processed_columns(dungeon_frame)
    classify(dungeon_frame, method)

def teste():
    initialize_data_frame()

    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()

    # 🔥 CONVERTE PARA TUPLA AQUI
    profiles = user_frame['PreFormAnswers'].apply(tuple)

    print("Combinações únicas:")
    print(sorted(profiles.unique()))

    print("\nQuantidade de combinações únicas:")
    print(profiles.nunique())

    print("\nFrequência de cada combinação:")
    print(profiles.value_counts())

    dungeon_frame = pd.merge(
        dungeon_frame,
        user_frame[['PlayerId', 'PreFormAnswers']],
        on=['PlayerId'],
        how='left'
    )

    print("\nFrequência no dungeon_frame final:")
    print(dungeon_frame['PreFormAnswers'].value_counts())

    # Listar todas as colunas
    colunas = dungeon_frame.columns.tolist()

    print(f"\nTotal de colunas: {len(colunas)}\n")

    # Exibir em ordem alfabética numerada
    for i, col in enumerate(sorted(colunas), 1):
        print(f"{i:3d}. {col}")


In [ ]:
teste()

Combinações únicas:
[(1, 3, 2, 4), (2, 1, 3, 4), (2, 1, 4, 3), (3, 2, 1, 4), (3, 2, 4, 1), (4, 1, 3, 2), (4, 2, 1, 3), (4, 3, 2, 1)]

Quantidade de combinações únicas:
8

Frequência de cada combinação:
PreFormAnswers
(4, 2, 1, 3)    3
(3, 2, 4, 1)    3
(1, 3, 2, 4)    3
(3, 2, 1, 4)    2
(4, 3, 2, 1)    1
(4, 1, 3, 2)    1
(2, 1, 3, 4)    1
(2, 1, 4, 3)    1
Name: count, dtype: int64

Frequência no dungeon_frame final:
PreFormAnswers
[1, 3, 2, 4]    26
[4, 1, 3, 2]    25
[2, 1, 3, 4]    24
[4, 2, 1, 3]    18
[3, 2, 1, 4]    11
[3, 2, 4, 1]     9
[2, 1, 4, 3]     5
[4, 3, 2, 1]     1
Name: count, dtype: int64

Total de colunas: 57

  1. CompletedAchievementQuests
  2. CompletedCreativityQuests
  3. CompletedDamageQuests
  4. CompletedExchangeQuests
  5. CompletedExploreQuests
  6. CompletedGatherQuests
  7. CompletedGiveQuests
  8. CompletedGoToQuests
  9. CompletedImmersionQuests
 10. CompletedKillQuests
 11. CompletedListenQuests
 12. CompletedMasteryQuests
 13. CompletedQuests
 14. C

In [ ]:
main(12)

/tmp/ipykernel_3158/2332524416.py:203: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _data_frame.replace([np.inf, -np.inf], 0, inplace=True)


Usuários que jogaram mais de um nível:
PlayerId
53627948       4
78194049       3
81972292      11
121412094     12
282268519      4
1093542367    12
1153731241     4
1776427228     2
2079201442     4
2079942857    12
2096844718     3
Name: LevelName, dtype: int64

Detalhamento:

Usuário 53627948 jogou:
['M:10T:20G:40E:30', 'M:20T:30G:40E:10', 'M:30T:10G:40E:20', 'M:40T:10G:30E:20']
--------------------------------------------------
Usuário 78194049 jogou:
['M:10T:20G:40E:30', 'M:10T:40G:30E:20', 'M:40T:10G:30E:20']
--------------------------------------------------
Usuário 81972292 jogou:
['M:10T:20G:30E:40', 'M:10T:20G:40E:30', 'M:20T:30G:10E:40', 'M:20T:30G:40E:10', 'M:20T:40G:10E:30', 'M:30T:10G:20E:40', 'M:30T:10G:40E:20', 'M:30T:40G:20E:10', 'M:40T:10G:30E:20', 'M:40T:20G:10E:30', 'M:40T:30G:20E:10']
--------------------------------------------------
Usuário 121412094 jogou:
['M:10T:20G:30E:40', 'M:10T:20G:40E:30', 'M:10T:40G:30E:20', 'M:20T:30G:10E:40', 'M:20T:30G:40E:10', 'M:20

In [ ]:
#TODO ORGANIZAR CASOS, PQ ESSE É MAIS ORGANZIAO DO QUE TRAMOP
#1º ELE TA LENDO OS DADOS
#2º TA ORGANIZANDO - DE FORMA (próprias entradas das dungeons, se funcionasse bem com os dados que temos, poderíamos deixar mais "granular", ao invés de falar A=1, C=2, etc)
# PEGANDO ID, A=1.., AVALICAO FORMS
#VER POR ITEM. VER POR USUARIO
#COSEENO E PEARSON

# PEGANDO ID, A=1.., COMPORATEMTNO (DESMENHO...)
#VER POR ITEM. VER POR USUARIO
#COSEENO E PEARSON

#3º ORGANIZAR DE FORMA - Poderíamos falar: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidade. E aí fazer essas combinações de colunas, que deixaria muito maior.


#4º E poderia fazer uma tabela (uma filtragem) para cada atributo.
#Faria outra tabela pros inimigos: Cada coluna uma dificuldade de inimigos (10, 12, 14, etc.)
#Outra poderia fazer pras quests... que aí sim poderiam ser as preferências de matar inimigo, explorar dungeon, etc.
#E aí recomendaria cada trecho separadamente...
#Pelo sistema ser nosso, a gnt pode expandir conforme achar melhor, ou manter assim por agora.


Entrada específica  (ex: A=4, C=2, M=1, I=3)
- baseada no usuario
  - ver similiareida dos itens
    - cosseno
    - pearson
  - Faria outra tabela pros inimigos: Cada coluna uma dificuldade de inimigos (10, 12, 14, etc.) Outra poderia fazer pras quests... que aí sim poderiam ser as preferências de matar inimigo, explorar dungeon, etc.
  )
- baseada no item (nota)
  - ver similiareida dos itens
    - cosseno (ok)
    - pearson (ok)
  - recomendar pro usuario nivel (ac...) -> so compara similarid entre niveis, mas nao fiz reocmencadp
  - fazer tabela do exmeplo tb
  - Faria outra tabela pros inimigos: Cada coluna uma dificuldade de inimigos (10, 12, 14, etc.) Outra poderia fazer pras quests... que aí sim poderiam ser as preferências de matar inimigo, explorar dungeon, etc.

Entrada granular (ex: 20 salas, 4 chaves, 1.5 linearidade)
para 2 e 3 segue os usbptrocessos
- baseada no usuario
  - ver similiareida dos itens
    - cosseno
    - pearson
  - Faria outra tabela pros inimigos: Cada coluna uma dificuldade de inimigos (10, 12, 14, etc.) Outra poderia fazer pras quests... que aí sim poderiam ser as preferências de matar inimigo, explorar dungeon, etc.
- baseada no item (nota)
  - ver similiareida dos itens
    - cosseno
    - pearson
  - Faria outra tabela pros inimigos: Cada coluna uma dificuldade de inimigos (10, 12, 14, etc.) Outra poderia fazer pras quests... que aí sim poderiam ser as preferências de matar inimigo, explorar dungeon, etc.